Telecom Customer Churn Predictive Pipeline & Account Segmentation

In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
print(f"Data loaded successfully. Shape: {df.shape}")

Data loaded successfully. Shape: (7043, 21)


In [5]:
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan).astype(float)
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

X = pd.get_dummies(df.drop(columns=['customerID', 'Churn']), drop_first=True)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [6]:
model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10, class_weight='balanced_subsample')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"Optimized Test Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred))

Optimized Test Accuracy: 76.79%
              precision    recall  f1-score   support

           0       0.89      0.78      0.83      1035
           1       0.55      0.73      0.63       374

    accuracy                           0.77      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.77      0.78      1409



In [7]:
probabilities = model.predict_proba(X_test)[:, 1]
segment_df = pd.DataFrame({'true_status': y_test.values, 'churn_probability': probabilities, 'tenure': X_test['tenure']})

conditions = [
    (segment_df['churn_probability'] >= 0.60),
    (segment_df['churn_probability'] < 0.30) & (segment_df['tenure'] >= 24),
    (segment_df['churn_probability'] < 0.60) & (segment_df['tenure'] < 6)
]
segment_df['customer_segment'] = np.select(conditions, ['At Risk', 'Loyal', 'Dormant'], default='Standard Active')
print(segment_df['customer_segment'].value_counts())

customer_segment
Loyal              549
At Risk            387
Standard Active    377
Dormant             96
Name: count, dtype: int64
